# 05 — Comparative Analysis

Compare AD vs Control samples: differential expression and gene set activity analysis.

In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd

from src.analysis import DifferentialExpression
from src.annotation import ADGeneSetAnalyzer
from src.visualization import Visualizer
from src.utils import load_config

config = load_config('config/config.yaml')
adata = sc.read_h5ad('data/processed/04_annotated.h5ad')

In [ ]:
# Check condition metadata
if 'condition' in adata.obs:
    print('Conditions:', adata.obs['condition'].value_counts())
else:
    print('No condition column found. Add one to run DE analysis.')

In [ ]:
# AD gene set scoring
ad_analyzer = ADGeneSetAnalyzer(adata)
scores = ad_analyzer.score_gene_sets()

# Summary per cell type
summary = ad_analyzer.get_gene_set_summary()
display(summary)

# Heatmap of gene set activity by cell type
sc.pl.heatmap(adata, var_names=scores.columns.tolist(), groupby='cell_type')

In [ ]:
# If condition available, run DE
if 'condition' in adata.obs:
    de = DifferentialExpression(adata)
    result = de.run_de(
        groupby='condition',
        group1='Control',
        group2='AD',
        method=config['differential_expression']['method'],
    )
    
    # Filter significant
    sig = result[result['pvals_adj'] < 0.05].copy()
    sig = sig[sig['logfoldchanges'].abs() > 0.25]
    print(f'Significant DE genes: {len(sig)}')
    display(sig.head(10))